In [7]:
import pandas as pd
df = pd.read_csv('city_day.csv')

In [8]:
df.shape
df.info()
df.describe()
df['City'].nunique()
df['AQI_Bucket'].unique()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 29531 entries, 0 to 29530
Data columns (total 16 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   City        29531 non-null  object 
 1   Date        29531 non-null  object 
 2   PM2.5       24933 non-null  float64
 3   PM10        18391 non-null  float64
 4   NO          25949 non-null  float64
 5   NO2         25946 non-null  float64
 6   NOx         25346 non-null  float64
 7   NH3         19203 non-null  float64
 8   CO          27472 non-null  float64
 9   SO2         25677 non-null  float64
 10  O3          25509 non-null  float64
 11  Benzene     23908 non-null  float64
 12  Toluene     21490 non-null  float64
 13  Xylene      11422 non-null  float64
 14  AQI         24850 non-null  float64
 15  AQI_Bucket  24850 non-null  object 
dtypes: float64(13), object(3)
memory usage: 3.6+ MB


array([nan, 'Poor', 'Very Poor', 'Severe', 'Moderate', 'Satisfactory',
       'Good'], dtype=object)

In [9]:
pip install meteostat geopy requests

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: c:\Users\prasi\.pyenv\pyenv-win\versions\3.10.11\python.exe -m pip install --upgrade pip


In [10]:

import pandas as pd
import requests
import time

# Load your dataset
df = pd.read_csv('city_day.csv')
df['Date'] = pd.to_datetime(df['Date'])

# COMPLETE Dictionary of Coordinates for all cities in your dataset
city_coords = {
    'Ahmedabad': {'lat': 23.0225, 'lon': 72.5714},
    'Aizawl': {'lat': 23.7271, 'lon': 92.7176},
    'Amaravati': {'lat': 16.5131, 'lon': 80.5165},
    'Amritsar': {'lat': 31.6340, 'lon': 74.8723},
    'Bengaluru': {'lat': 12.9716, 'lon': 77.5946},
    'Bhopal': {'lat': 23.2599, 'lon': 77.4126},
    'Brajrajnagar': {'lat': 21.8216, 'lon': 83.9216},
    'Chandigarh': {'lat': 30.7333, 'lon': 76.7794},
    'Chennai': {'lat': 13.0827, 'lon': 80.2707},
    'Coimbatore': {'lat': 11.0168, 'lon': 76.9558},
    'Delhi': {'lat': 28.6139, 'lon': 77.2090},
    'Ernakulam': {'lat': 9.9816, 'lon': 76.2999},
    'Gurugram': {'lat': 28.4601, 'lon': 77.0199},
    'Guwahati': {'lat': 26.1445, 'lon': 91.7362},
    'Hyderabad': {'lat': 17.3850, 'lon': 78.4867},
    'Jaipur': {'lat': 26.9124, 'lon': 75.7873},
    'Jorapokhar': {'lat': 23.7004, 'lon': 86.4125},
    'Kochi': {'lat': 9.9312, 'lon': 76.2673},
    'Kolkata': {'lat': 22.5726, 'lon': 88.3639},
    'Lucknow': {'lat': 26.8467, 'lon': 80.9462},
    'Mumbai': {'lat': 19.0760, 'lon': 72.8777},
    'Patna': {'lat': 25.5941, 'lon': 85.1376},
    'Shillong': {'lat': 25.5788, 'lon': 91.8933},
    'Talcher': {'lat': 20.9509, 'lon': 85.2163},
    'Thiruvananthapuram': {'lat': 8.5241, 'lon': 76.9366},
    'Visakhapatnam': {'lat': 17.6868, 'lon': 83.2185}
}

weather_frames = []

# Fetch data using the complete list
for city in df['City'].unique():
    # Safety check in case a new city appears in future data
    if city not in city_coords:
        print(f"⚠️ Skipping {city}: Coordinates still missing.")
        continue
        
    lat = city_coords[city]['lat']
    lon = city_coords[city]['lon']
    
    # Get date range for this specific city
    city_data = df[df['City'] == city]
    start_date = city_data['Date'].min().strftime('%Y-%m-%d')
    end_date = city_data['Date'].max().strftime('%Y-%m-%d')
    
    print(f"Fetching data for {city}...")
    
    try:
        # Fetch Temperature, Humidity, and Wind Speed
        url = f"https://archive-api.open-meteo.com/v1/archive?latitude={lat}&longitude={lon}&start_date={start_date}&end_date={end_date}&daily=temperature_2m_mean,relative_humidity_2m_mean,wind_speed_10m_max&timezone=auto"
        
        response = requests.get(url)
        
        if response.status_code == 200:
            data = response.json()
            
            # Create a temporary dataframe for the weather data
            temp_df = pd.DataFrame({
                'Date': pd.to_datetime(data['daily']['time']),
                'City': city,
                'Temp_Mean': data['daily']['temperature_2m_mean'],
                'Humidity_Mean': data['daily']['relative_humidity_2m_mean'],
                'Wind_Speed_Max': data['daily']['wind_speed_10m_max']
            })
            weather_frames.append(temp_df)
        else:
            print(f"❌ Error fetching {city}: {response.status_code}")
            
    except Exception as e:
        print(f"❌ Exception for {city}: {e}")
    
    # Pause briefly to respect API rate limits
    time.sleep(1)

# Combine and Merge
if weather_frames:
    all_weather_df = pd.concat(weather_frames)
    
    # Merge with original data
    final_df = pd.merge(df, all_weather_df, on=['City', 'Date'], how='left')
    
    # Save to CSV
    final_df.to_csv('city_day_with_weather_complete.csv', index=False)
    print("\n✅ Success! All cities updated. Saved to 'city_day_with_weather_complete.csv'")
else:
    print("\n❌ No weather data fetched.")

Fetching data for Ahmedabad...
Fetching data for Aizawl...
Fetching data for Amaravati...
Fetching data for Amritsar...
Fetching data for Bengaluru...
Fetching data for Bhopal...
Fetching data for Brajrajnagar...
Fetching data for Chandigarh...
Fetching data for Chennai...
Fetching data for Coimbatore...
Fetching data for Delhi...
Fetching data for Ernakulam...
Fetching data for Gurugram...
Fetching data for Guwahati...
Fetching data for Hyderabad...
Fetching data for Jaipur...
Fetching data for Jorapokhar...
Fetching data for Kochi...
Fetching data for Kolkata...
Fetching data for Lucknow...
Fetching data for Mumbai...
Fetching data for Patna...
Fetching data for Shillong...
Fetching data for Talcher...
Fetching data for Thiruvananthapuram...
Fetching data for Visakhapatnam...

✅ Success! All cities updated. Saved to 'city_day_with_weather_complete.csv'


In [11]:
import pandas as pd

# Load the dataset
df = pd.read_csv('city_day_with_weather_complete.csv')

# Remove rows where AQI or AQI_Bucket is null
df_cleaned = df.dropna(subset=['AQI', 'AQI_Bucket'])

# Save the cleaned dataset to a new file
df_cleaned.to_csv('city_day_cleaned.csv', index=False)

In [12]:
import pandas as pd

# 1. Load the dataset
df = pd.read_csv('city_day_cleaned.csv')

# 2. Convert 'Date' to datetime objects to ensure correct chronological order
df['Date'] = pd.to_datetime(df['Date'])

# 3. Define a function to interpolate missing values within a specific group (City)
def interpolate_group(group):
    # Select only numeric columns for interpolation
    numeric_cols = group.select_dtypes(include=['number']).columns
    
    # Apply linear interpolation
    # limit_direction='both' ensures missing values at the start/end are also filled
    group[numeric_cols] = group[numeric_cols].interpolate(method='linear', limit_direction='both')
    
    return group

# 4. Apply the interpolation function to each city independently
# group_keys=False prevents pandas from adding an extra index layer
df_interpolated = df.groupby('City', group_keys=False).apply(interpolate_group)

# 5. Save the result to a new CSV file
df_interpolated.to_csv('city_day_interpolated.csv', index=False)

# Optional: Check remaining missing values
print(df_interpolated.isnull().sum())

C:\Users\prasi\AppData\Local\Temp\ipykernel_6700\1705854046.py:22: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_interpolated = df.groupby('City', group_keys=False).apply(interpolate_group)


City                  0
Date                  0
PM2.5                 0
PM10               1893
NO                    0
NO2                   0
NOx                 771
NH3                1334
CO                    0
SO2                   0
O3                  153
Benzene            2259
Toluene            4084
Xylene            12381
AQI                   0
AQI_Bucket            0
Temp_Mean             0
Humidity_Mean         0
Wind_Speed_Max        0
dtype: int64
